# AWF Neural Adapter — Colab QLoRA (experimental)

**Notebook de ejecución opcional, no validación científica.** Se usa únicamente el dataset sintético generado en la sesión. No monta Drive ni sube documentos; no publica pesos. Debes activar GPU en Runtime antes de ejecutarlo. Los recursos de Colab no están garantizados.

## 1. Recuperar exactamente la rama experimental

Esta celda descarga código público de AWF. Evita cargar manuscritos privados o contenido histórico de la suite.

In [ ]:
import pathlib
import subprocess
import sys

root = pathlib.Path("/content/awf")
if root.exists():
    raise RuntimeError("Ruta de proyecto existente; no sobrescribir")
subprocess.check_call(
    [
        "git",
        "clone",
        "--depth",
        "1",
        "--branch",
        "feat/s3-peft-training-foundation",
        "https://github.com/David-std/AWF.git",
        str(root),
    ]
)
subprocess.check_call(
    [sys.executable, "-m", "pip", "install", "-q", "-r", str(root / "requirements-neural.txt")]
)
sys.path.insert(0, str(root / "src"))
print(
    "Git revision:",
    subprocess.check_output(["git", "-C", str(root), "rev-parse", "HEAD"], text=True).strip(),
)

## 2. Comprobar el acelerador y crear datos originales ficticios

No son documentos académicos ni un holdout independiente. No utilizar para medir fidelidad.

In [ ]:
import json

import torch

from awf.training.synthetic_smoke import make_private_smoke_dataset

if not torch.cuda.is_available():
    raise RuntimeError("CUDA GPU no disponible; no ejecutar entrenamiento")
print("GPU:", torch.cuda.get_device_name(0))
folder = root / ".awf" / "neural-smoke-colab"
digest = make_private_smoke_dataset(folder)
print("Manifest SHA-256:", digest)
print("Inputs:", sorted(x.name for x in folder.iterdir()))

## 3. Preflight y ejecución deliberada

La opción de entrenamiento está desactivada por defecto. Actívala manualmente después de revisar el dataset y la cuota. Los pesos quedarán únicamente en la sesión Colab bajo `.awf/model-runs/`.

In [ ]:
import os

from awf.training.runner import main as training_main

os.chdir(root)
args = [
    "--manifest",
    str(folder / "manifest.json"),
    "--manifest-sha256",
    digest,
    "--profile",
    str(root / "config/training/colab_qlora_0_5b.json"),
    "--allow-synthetic-smoke",
]
assert training_main(args) == 0
EXECUTE_TRAINING = False
if EXECUTE_TRAINING:
    assert training_main(args + ["--execute"]) == 0
else:
    print("Solo preflight. No se entrenó ningún adaptador.")

## 4. Verificar artefactos privados

Después de activar el entrenamiento, comprueba que el recibo informe actualizaciones de gradiente y hashes de archivos. La pérdida de entrenamiento **no** demuestra fidelidad semántica. Nunca utilizar un conjunto sintético como publicación científica. El notebook no se ha ejecutado en Colab durante su generación.

In [ ]:
receipts = (
    sorted((root / ".awf" / "model-runs").glob("*/receipt.json"))
    if (root / ".awf" / "model-runs").exists()
    else []
)
for filename in receipts:
    report = json.loads(filename.read_text(encoding="utf-8"))
    print(
        {
            "receipt": str(filename),
            "status": report["status"],
            "gradient_updates": report["gradient_updates_observed"],
            "semantic_verified": report["semantic_equivalence_verified"],
        }
    )
if not receipts:
    print("No existe adaptador: solo preparación o entrenamiento omitido.")